# 01 基础：SQL 与存储

## 八股问答

### 1. 增删改查

查询关键字的逻辑执行顺序：

```text
FROM → JOIN / ON → WHERE → GROUP BY → HAVING → SELECT → DISTINCT → ORDER BY → LIMIT
```

#### LeetCode 题目补充

通过 LeetCode SQL 题补充练习。
### 2. 数据库三大设计范式

#### 第一范式（1NF）：字段不可再分

1NF 要求一个字段只保存一个原子的值，不能把一个字段当成列表或逗号分隔串使用。

反例：`users.phone` 同时保存 `138xxx,139xxx`。这个字段里实际有两个电话号码，
查询其中一个、更新其中一个、对号码做唯一约束都要先拆字符串，容易出错。

如果一位用户最多只有一个手机号，直接保存为单个字段即可，不需要拆表：

```text
users(id, phone)
```

只有当业务确实允许一位用户拥有多个电话号码时，才拆成独立表：

```text
users(id)
user_phones(user_id, phone)
```

拆表不是因为 1NF 要求“一定拆表”，而是因为“一个实体有多个同类值”时，把多个值拆成
多行，才能在每行只放一个号码，方便查询、更新和唯一约束。如果字段只是本身复杂但仍是
一个整体值，应按实际语义判断是否拆，而不是看到“多个值”就机械拆表。

#### 第二范式（2NF）：消除部分依赖

在满足 1NF 的基础上，非主属性必须完全依赖整个候选键，不能只依赖联合主键的一部分。

例如，选课表以 `(student_id, course_id)` 为联合主键，表中同时保存了
`student_name` 和 `course_name`。`student_name` 只依赖 `student_id`，
`course_name` 只依赖 `course_id`，它们都没有完全依赖整个联合主键：

```text
course_selections(student_id, course_id, student_name, course_name, score)
```

应拆成三张表：

```text
students(student_id, student_name)
courses(course_id, course_name)
course_selections(student_id, course_id, score)
```

如果不去掉这两个冗余字段，同一个学生选了 50 门课，`student_name` 就会在 50 行里
重复 50 次。学生改名必须更新所有选课行，容易漏改；新学生还没有选课时，也无法只插入
学生信息而不先造一条选课记录。`course_name` 也有同样问题，只是依赖的是 `course_id`。

#### 第三范式（3NF）：消除传递依赖

在满足 2NF 的基础上，非主属性不能依赖其他非主属性，只能直接依赖候选键。

例如，`users(id, department_id, department_name)` 中，部门名称依赖部门 ID，
应拆成：

```text
users(id, department_id)
departments(department_id, department_name)
```

#### 不满足范式会有什么问题？

- **违反 1NF**：一个字段塞了多个值，查询单一值、更新单一值和唯一约束都很麻烦，
  容易靠字符串拆解处理，还会出现格式不一致和脏数据。
- **违反 2NF**：`student_name` 只依赖 `student_id`、`course_name` 只依赖
  `course_id`，却在每个选课行重复保存；改名要改大量选课行，新学生或新课程在尚无关联
  记录时也难以独立插入。
- **违反 3NF**：`department_name` 只依赖 `department_id` 却重复出现在每个员工行里，
  部门改名要更新很多行；新部门没有员工时，也需要靠占位数据才能插入。

这些问题的共同点是同一个事实被重复保存，导致插入、更新、删除时可能出现数据不一致。
反范式可以为了查询性能保留冗余，但必须明确冗余数据的来源和维护方式。

范式用于减少数据冗余以及插入、更新、删除异常。实际系统可为查询性能适度反范式，
但必须明确冗余字段的数据来源和一致性维护方式。

### 3. SELECT 语句的执行流程？

一条查询经过五个环节：

1. 连接器
   - 建立连接：TCP 握手、账号密码校验。
   - 权限校验：连接时校验账号权限，执行时再校验库、表、列权限。
   - 管理连接和线程：控制连接生命周期和线程分配。
2. 解析器
   - 词法分析：把 SQL 拆成关键字、表名、字段名等 token。
   - 语法分析：按 MySQL 语法生成解析树，语法错误在这里报。
   - 语义检查：校验表、字段是否存在，权限是否足够。
3. 优化器
   - 根据统计信息估算代价，选择执行计划。
   - 决定走哪个索引、join 顺序，是否排序或使用临时表。
   - 只输出计划，还没有真正读数据。
4. 执行器
   - 按执行计划调用存储引擎接口（handler API）逐行取数据。
   - 负责服务层的过滤、排序、聚合、投影，并组织结果集。
5. 存储引擎（InnoDB）
   - 接收执行器的读取请求，先在 Buffer Pool 里找页。
   - 未命中再从磁盘读页，返回符合条件的行。
   - 真正发生磁盘 IO 的是这一层，再往下就是文件系统。

这些环节对应 MySQL 的分层架构：

| 架构分层 | 对应的执行环节 |
| --- | --- |
| 连接层 | 连接器 |
| 服务层 | 解析器、优化器、执行器 |
| 存储引擎层 | InnoDB 等存储引擎 |
| 存储层 | 文件系统 / 磁盘 |

服务层是解析、优化、执行所在的位置，不是独立的一个环节；存储层是引擎
最终读写数据的物理载体，不属于 MySQL 的功能模块。所以执行流程按环节
讲更具体，按分层讲只能说明“在哪个层”。

MySQL 8.0 删除了查询缓存，所以不用再回答“命中缓存直接返回”。面试说清
“解析 → 优化 → 执行”三层即可，连接器和存储引擎是前后两端。

### 4. 一行记录在 InnoDB 里是如何存储的？

InnoDB 默认 `DYNAMIC` 行格式，一行记录在页内大致分四段：

- 变长字段长度列表：记录变长列的长度，NULL 列不占。
- NULL 标志位：用位图标记哪些列是 NULL，节省存储。
- 记录头信息：记录删除标记、下一条记录位置等，约 5 字节。
- 隐藏列：`DB_ROW_ID`（无主键时）、`DB_TRX_ID`（最近修改事务 ID）、
  `DB_ROLL_PTR`（回滚指针）。
- 数据列：按建表顺序存放实际字段值。

数据按主键顺序存放在 16KB 的页里，页头记录页内记录数，页尾存校验和。
理解“一行 = 字段值 + 隐藏列 + 记录头”就够回答大多数问题。

### 5. 执行 UPDATE 会发生什么？

一条 UPDATE 是“读 + 写”的完整链路：

1. 定位：走索引找到目标行（当前读）。
2. 加锁：对目标记录加 X 锁，RR 下是临键锁。
3. 写 undo：把旧版本写入 undo log，形成版本链，保证可回滚。
4. 改内存：在 Buffer Pool 中修改数据页，页标记为脏页。
5. 记 redo：按 WAL 先写 redo log，保证崩溃可恢复。
6. 记 binlog：服务层记录逻辑变更，供复制和恢复。
7. 两阶段提交：redo 先 prepare，binlog 写成功后再 commit，保证两者一致。
8. 提交：释放锁；脏页由后台线程择机刷盘。

“先写日志再改数据”是 WAL 的核心，后面日志篇会展开。


## 面试话术

“一条 SELECT 经过连接器、解析器、优化器、执行器，最后调 InnoDB 接口；一条
UPDATE 多了加锁、写 undo、改 Buffer Pool、记 redo 和 binlog、两阶段提交这几步。
我会用 EXPLAIN 看优化器的计划，用 EXPLAIN ANALYZE 看真实执行时间，出现明显
偏差时优先怀疑统计信息或索引选择。”


## 自查清单

- [ ] 能写出 SELECT 的书写顺序和执行顺序
- [ ] 能说出查询经过的五个环节
- [ ] 能画出 DYNAMIC 行格式的四段结构
- [ ] 能按顺序说出 UPDATE 的完整链路
- [ ] 能在终端跑 CRUD 并用 EXPLAIN ANALYZE 看真实执行
